
# ETL — Análise dos dados de despesas assistenciais da ANS

Este notebook prepara uma **tabela analítica** a partir de dados públicos da **ANS (Agência Nacional de Saúde Suplementar)** para posterior uso em modelos de detecção de anomalias (ex.: Isolation Forest, Autoencoders).  


> **Como usar**  
> 1) Preencha `DATA_URLS` com 1..N URLs (ou paths locais) dos CSVs da ANS.  
> 2) Execute as células em ordem.  
> 3) Ao final, você terá um arquivo `analitico_anomalias.parquet` pronto para modelagem.


In [1]:
!pip install -qq fastparquet

In [2]:
# ==== Configuração Geral ====
from pathlib import Path
import os
import glob

# ==== Utilitários ====
import io, re, zipfile
import pandas as pd
import numpy as np

import requests
from time import sleep
import shutil


In [35]:
BASE_URL = "https://dadosabertos.ans.gov.br/FTP/PDA/demonstracoes_contabeis/"

In [36]:
path_downloads = os.getenv("PATH_DOWNLOADS")
FOLDER_RAW = "../../bases/despesas_contabeis/data/raw"


In [37]:
if not path_downloads:
    raise RuntimeError("Variável de ambiente PATH_DOWNLOADS não encontrada. Faça: export PATH_DOWNLOADS=/caminho")

DOWNLOAD_DIR = Path(path_downloads)
RAW_DIR = Path(FOLDER_RAW)

In [38]:
def fetch_text(url: str, timeout=60) -> str:
    """
    Faz uma requisição HTTP GET para a URL informada e retorna o conteúdo como texto.

    Args:
        url (str): URL do recurso a ser acessado.
        timeout (int, opcional): Tempo limite para a requisição em segundos. Padrão é 60.

    Returns:
        str: Conteúdo da resposta em formato texto.

    Raises:
        requests.HTTPError: Se a resposta HTTP indicar erro.
        requests.RequestException: Para outros erros de requisição.
    """
    
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.text

In [39]:
def list_year_dirs(base_url: str):
    """
    Lista os anos disponíveis no diretório base da ANS.

    Args:
        base_url (str): URL base do diretório de demonstrações contábeis.

    Returns:
        list[int]: Lista ordenada dos anos disponíveis encontrados no diretório.
    """
    html = fetch_text(base_url)
    years = sorted(set(int(y) for y in re.findall(r'>(\d{4})/</a>', html)))
    return years

In [40]:
def list_zip_urls_for_year(base_url: str, year: int):
    """
    Lista todas as URLs de arquivos ZIP disponíveis para um determinado ano no diretório da ANS.

    Args:
        base_url (str): URL base do diretório de demonstrações contábeis.
        year (int): Ano para o qual os arquivos ZIP devem ser listados.

    Returns:
        list[str]: Lista de URLs completas dos arquivos ZIP encontrados para o ano informado.
    """
    html = fetch_text(f"{base_url}{year}/")
    zips = re.findall(r'href="([^"]+\.zip)"', html)
    return [f"{base_url}{year}/{z}" for z in zips]

In [41]:
def stream_download(url: str, dest_path: Path, retries: int = 3):
    """
    Faz o download de um arquivo de uma URL para um caminho local usando streaming,
    com suporte a múltiplas tentativas em caso de falha.

    Args:
        url (str): URL do arquivo a ser baixado.
        dest_path (Path): Caminho local onde o arquivo será salvo.
        retries (int, opcional): Número de tentativas em caso de erro. Padrão é 3.

    Raises:
        Exception: Se todas as tentativas de download falharem ou se o tamanho do arquivo baixado for diferente do esperado.
    """
    tmp_path = dest_path.with_suffix(dest_path.suffix + ".part")
    for attempt in range(1, retries + 1):
        try:
            with requests.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                total = int(r.headers.get("Content-Length", 0)) or None

                with open(tmp_path, "wb") as f:
                    downloaded = 0
                    for chunk in r.iter_content(chunk_size=1024 * 256):
                        if chunk:
                            f.write(chunk)
                            downloaded += len(chunk)

                # Verifica tamanho se houver Content-Length
                if total and tmp_path.stat().st_size != total:
                    tmp_path.unlink(missing_ok=True)
                    raise IOError("Tamanho inesperado no download.")

                tmp_path.replace(dest_path)  # rename para o final
                return  # sucesso
        except Exception:
            if attempt == retries:
                tmp_path.unlink(missing_ok=True)
                raise
            sleep(1.5 * attempt)

In [42]:
def already_same_size(file_path: Path, url: str) -> bool:
    """
    Verifica se o arquivo local possui o mesmo tamanho do arquivo remoto disponível na URL.

    Args:
        file_path (Path): Caminho do arquivo local a ser verificado.
        url (str): URL do arquivo remoto para comparação.

    Returns:
        bool: True se ambos os arquivos existem e possuem o mesmo tamanho, False caso contrário.

    Observações:
        - Utiliza uma requisição HTTP HEAD para obter o tamanho do arquivo remoto.
        - Se a requisição falhar, retorna False sem impedir o download futuro.
    """
    
    try:
        head = requests.head(url, timeout=30)
        head.raise_for_status()
        remote_size = int(head.headers.get("Content-Length", "0"))
        return remote_size > 0 and file_path.exists() and file_path.stat().st_size == remote_size
    except Exception:
        # se HEAD falhar, não impede o download
        return False



In [43]:
YEARS = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
years = YEARS or list_year_dirs(BASE_URL)

downloaded = []
moved = []


for y in years:
    urls = list_zip_urls_for_year(BASE_URL, y)
    if not urls:
        print(f"[{y}] Nenhum ZIP encontrado.")
        continue
    
    print(f"[{y}] {len(urls)} arquivos ZIP encontrados.")

    for url in urls:
        # nome do arquivo (ex.: 1T2024.zip)
        fname = url.rstrip("/").split("/")[-1]
        local_tmp = DOWNLOAD_DIR / fname
        final_dest = RAW_DIR / fname

        if final_dest.exists() and already_same_size(final_dest, url):
            print(f"✓ Já existe (ok): {final_dest.name}")
            continue

        if local_tmp.exists() and already_same_size(local_tmp, url):
            print(f"↪ Arquivo já no downloads (ok): {local_tmp.name}")

        else:
            print(f"↓ Baixando: {url}")
            stream_download(url, local_tmp)
            downloaded.append(str(local_tmp))

        # Move para o destino final (sobrescreve se for diferente)
        if final_dest.exists():
            # Se tamanhos divergem, substitui
            if final_dest.stat().st_size != local_tmp.stat().st_size:
                final_dest.unlink()
                shutil.move(str(local_tmp), str(final_dest))
                print(f"→ Atualizado: {final_dest}")
                moved.append(str(final_dest))
            else:
                # Mesmo tamanho: remove duplicata do downloads
                local_tmp.unlink(missing_ok=True)
        else:
            shutil.move(str(local_tmp), str(final_dest))
            print(f"→ Movido: {final_dest}")
            moved.append(str(final_dest))

print("\nResumo:")
print(f"- Baixados: {len(downloaded)}")
print(f"- Movidos/Atualizados: {len(moved)}")
print(f"- Destino: {RAW_DIR.resolve()}")



[2015] 4 arquivos ZIP encontrados.
↪ Arquivo já no downloads (ok): 1T2015.zip
→ Movido: ../../bases/despesas_contabeis/data/raw/1T2015.zip
↓ Baixando: https://dadosabertos.ans.gov.br/FTP/PDA/demonstracoes_contabeis/2015/2T2015.zip
→ Movido: ../../bases/despesas_contabeis/data/raw/2T2015.zip
↓ Baixando: https://dadosabertos.ans.gov.br/FTP/PDA/demonstracoes_contabeis/2015/3T2015.zip
→ Movido: ../../bases/despesas_contabeis/data/raw/3T2015.zip
↓ Baixando: https://dadosabertos.ans.gov.br/FTP/PDA/demonstracoes_contabeis/2015/4T2015.zip
→ Movido: ../../bases/despesas_contabeis/data/raw/4T2015.zip
[2016] 4 arquivos ZIP encontrados.
↓ Baixando: https://dadosabertos.ans.gov.br/FTP/PDA/demonstracoes_contabeis/2016/1T2016.zip
→ Movido: ../../bases/despesas_contabeis/data/raw/1T2016.zip
↓ Baixando: https://dadosabertos.ans.gov.br/FTP/PDA/demonstracoes_contabeis/2016/2T2016.zip
→ Movido: ../../bases/despesas_contabeis/data/raw/2T2016.zip
↓ Baixando: https://dadosabertos.ans.gov.br/FTP/PDA/demonstra

In [44]:
files_path = glob.glob('/mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/*.zip')
len(files_path)

41

In [54]:
def write_outputs(df: pd.DataFrame, path: str | Path):
    """Salva DataFrame em Parquet."""
    try:
        path_new = path.replace('.zip', '.parquet')
        
        df.to_parquet(path_new, index=False, engine='fastparquet')
        print(f"----> salvo: {path_new}")
        return True
    
    except Exception as e:
        print("### ERROR ###")
        print(e)
        print(f"----> erro ao salvar Parquet: {path_new}")
        print()
        return False
        

In [59]:
def load_source(src: str | Path):
    """Carrega um CSV (ou CSV dentro de ZIP) em DataFrame."""
    p = Path(src)
 
    # Se ZIP, tenta encontrar o primeiro CSV dentro
    if str(p).lower().endswith(".zip"):
        with zipfile.ZipFile(p, 'r') as zf:
            # escolhe o primeiro CSV
            csv_names = [n for n in zf.namelist() if n.lower().endswith('.csv')]
            if not csv_names:
                raise ValueError(f"ZIP sem CSV: {p}")
            with zf.open(csv_names[0]) as f:
                
                df = pd.read_csv(f, sep=';', encoding='latin1', low_memory=False)
                df['DESCRICAO'] = df['DESCRICAO'].astype(str)
                
                if write_outputs(df, str(p)):
                    return {"status":True,"msg":">>>>>> arquivo extraido e transformado com sucesso!!!"}
                else:
                    return {"status":False,"msg":">>>>>> falha ao salvar arquivo!!!"}



In [60]:
def process_all_files(files: list[str | Path]):
    """--> Processa todos os arquivos (CSV/ZIP) e salva os resultados."""
    files_error = []
    for f in files:
        print(f"Processando: {f}")
        result = load_source(f)
        if result['status']==False:
            #print(f)
            files_error.append(f)
    
    return {"Qtd files sucess":len(files)-len(files_error),
            "Qtd files error":len(files_error),
            "files_error":files_error}
        #print(result)

In [61]:
result_process = process_all_files(files_path)

Processando: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2015.zip
----> salvo: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2015.parquet
Processando: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2016.zip
----> salvo: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2016.parquet
Processando: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2017.zip
----> salvo: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2017.parquet
Processando: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2018.zip
----> salvo: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2018.parquet
Processando: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2019.zip
----> salvo: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2019.parquet
Processando: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2020.zip
----> salvo: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data

In [62]:
result_process

{'Qtd files sucess': 41, 'Qtd files error': 0, 'files_error': []}

In [63]:
for f in files_path:
    try:
        os.remove(f)
        print(f"Removido: {f}")
    except Exception as e:
        print(f"Erro ao remover {f}: {e}")

Removido: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2015.zip
Removido: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2016.zip
Removido: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2017.zip
Removido: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2018.zip
Removido: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2019.zip
Removido: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2020.zip
Removido: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2021.zip
Removido: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2022.zip
Removido: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2023.zip
Removido: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2024.zip
Removido: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/1T2025.zip
Removido: /mnt/e/portfolio/prj_ans/bases/despesas_contabeis/data/raw/2T2015.zip
Removido: /mnt/e/portfolio/prj_ans/bases